# 08 · Heldout (final test) evaluation

The final, untouched test: 1,066 prompts from Dataset2's 60 **heldout** seeds (representative
stratified sample; 635 pure English/Japanese, 431 code-mixed), answered by the original model, the
published restricted and broad adapters, and the layer-9 / layer-18 seeded adapters from the
layer-control study, and judged by Qwen3Guard. Plan written before the run:
[`docs/HELDOUT_PLAN.md`](../docs/HELDOUT_PLAN.md); results: [`docs/HELDOUT_FINDINGS.md`](../docs/HELDOUT_FINDINGS.md).

- **View the results (CPU, seconds):** run sections 1 and 3 with `RUN = False` (default).
- **Rerun (GPU: A100/G4/L4, about 2 h for 7 models; HF token with Tiny Aya access):** set `RUN = True`.
  The runner checkpoints every 128 prompts and resumes from its partial files.

> Content warning: jailbreak-style prompts; some answers are judged unsafe and shown verbatim.

## 1. Setup

In [ ]:
# Project setup (Colab or local): fetch this repository's code and small data files, then load
# the shared config. Large files come from the project's Hugging Face datasets on first use.
import subprocess
import sys
from pathlib import Path

if Path("/content").exists():                     # Google Colab
    REPO = Path("/content/tiny-aya-safety")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/ishane0620/tiny-aya-safety", str(REPO)], check=True)
else:                                             # local checkout: this notebook is in <repo>/notebooks
    REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from tiny_aya_safety.config import CONFIG, adapter_source, artifact, data_path, output_dir

OUT = output_dir()   # everything this notebook writes goes here (override with TAS_OUTPUT_DIR)
print("repo:", REPO, "| outputs:", OUT)

In [ ]:
import os
if Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate", "peft",
                    "torchao"], check=True)
    if not os.environ.get("HF_TOKEN"):
        try:
            from google.colab import userdata
            os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        except Exception:
            print("No HF_TOKEN secret found; the gated base model download will fail.")

## 2. Run the evaluation (GPU, optional)

In [ ]:
MODELS = ["baseline", "restricted", "broad", "L9_s0", "L9_s1", "L18_s0", "L18_s1"]
RUN = False   # True: regenerate everything; False: analyse the published results in results/heldout/

if RUN:
    cmd = [sys.executable, str(REPO / "scripts/run_heldout_eval.py"), "--models", *MODELS, "--out", str(OUT / "heldout")]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
        for line in p.stdout:
            if "it/s" not in line:
                print(line, end="")
    assert p.returncode == 0

## 3. Results

In [ ]:
answers_dir = (OUT / "heldout") if RUN else (REPO / "results/heldout")
subprocess.run([sys.executable, str(REPO / "scripts/analyze_heldout.py"), str(answers_dir)], check=True)
from IPython.display import HTML, Image, Markdown, display
display(Markdown((answers_dir / "HELDOUT_TABLES.md").read_text()))
display(Image(str(answers_dir / "heldout.png")))

## 4. Browse answers

`browse(model=..., set='pure'|'mixed', kind='harmful'|'twin', label='Unsafe', refused=1, framing=..., language=...)`; text is shown escaped.

In [ ]:
import pandas as pd
df = pd.concat([pd.read_json(p, lines=True) for p in sorted(answers_dir.glob("answers_*.jsonl"))], ignore_index=True)
STYLE = "<style>.ev td{white-space:pre-wrap;vertical-align:top;text-align:left;max-width:460px;font-size:12px}</style>"
COLS = ["model", "set", "language", "culture", "framing", "kind", "expected", "prompt", "answer", "label", "refused"]

def browse(max_rows=40, **f):
    d = df
    for k, v in f.items():
        d = d[d[k].isin(v if isinstance(v, (list, tuple, set)) else [v])]
    print(f"{len(d)} rows")
    display(HTML(STYLE + d[COLS].head(max_rows).to_html(index=False, escape=True, na_rep="-").replace("<table", '<table class="ev"', 1)))

browse(model="restricted", label="Unsafe")